# 02 — Read Source Blob via SAS Token
**Day 1 | Part 7.4**

Reads pre-loaded source data from the shared external Azure Blob Storage
(`dataenggdailystorage`) using a SAS token stored in **your** Key Vault.

---

## What is this notebook doing?

The project has **two different storage accounts**:

| Storage Account | Owner | What it holds | How you connect |
|---|---|---|---|
| `evdatalakedev` | You | Your Bronze / Silver / Gold data | SP OAuth (`abfss://`) |
| `dataenggdailystorage` | Course instructor | Raw source CSV files | SAS Token (`wasbs://`) |

This notebook connects to the **instructor's** storage account and reads source files from it.
You have **read + list only** access — you cannot write or delete anything there.

---

## Why `wasbs://` and not `abfss://`?

- `abfss://` = **Azure Data Lake Storage Gen2** protocol. Requires OAuth (Service Principal login). Only works when the storage account has **Hierarchical Namespace** enabled AND you have been given an RBAC role on it.
- `wasbs://` = **Windows Azure Storage Blob Service** protocol. Works with a SAS token. Simpler, no OAuth needed. Works on any Azure Blob Storage account.

Since `dataenggdailystorage` belongs to someone else and they gave you a SAS token (not an SP), you use `wasbs://`.

---

## Secrets to add in your Key Vault before running

| Secret Name | Value |
|---|---|
| `source-storage-account` | `dataenggdailystorage` |
| `source-container` | `source` |
| `source-sas-token` | Provided during the session |

## Cell 1 — Load credentials from Key Vault + configure Spark

**What this cell does:**
1. Reads 3 secrets from Key Vault — nothing is hardcoded in the notebook
2. Tells Spark: "when you see a path for `dataenggdailystorage`, use this SAS token to authenticate"

**Line by line:**

- `SCOPE = "kv-ev-scope"` — the name of the Databricks secret scope linked to your Key Vault. Acts as a named bridge: `"kv-ev-scope"` → your Key Vault → the actual secret value.
- `dbutils.secrets.get(scope=SCOPE, key="...")` — reads a secret from Key Vault without ever showing the value. Even if you `print()` it, Databricks shows `[REDACTED]`. The `key=` must match exactly the secret name you created in Key Vault.
- `STORAGE_ACCOUNT` — set to `"dataenggdailystorage"` (the external account, not yours).
- `CONTAINER` — set to `"source"` — the container name inside that storage account.
- `SAS_TOKEN` — the full token string: `se=2027-07-30&sp=rl&spr=https&...`. `sp=rl` means read + list permissions only.
- `spark.conf.set(key, value)` — sets a Spark config for this session only. Cleared on cluster restart.
- The key `fs.azure.sas.{container}.{account}.blob.core.windows.net` tells Spark: "when you access a `wasbs://source@dataenggdailystorage...` path, authenticate using this SAS token."

In [ ]:
SCOPE = "kv-ev-scope"

try:
    STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
    CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
    SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

    print(f"Storage account : {STORAGE_ACCOUNT}")
    print(f"Container       : {CONTAINER}")
    print(f"SAS token       : [REDACTED]")
    print("Credentials loaded from Key Vault — OK")

except Exception as e:
    print(f"ERROR: {e}")
    print("\nSecrets missing. Add these 3 secrets to Key Vault first:")
    print("  Portal → Key vaults → kv-ev-intelligence-dev → Secrets → + Generate/Import")
    print("  source-storage-account  →  dataenggdailystorage")
    print("  source-container        →  source")
    print("  source-sas-token        →  <value provided during session>")
    raise

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)
print("Spark SAS config set — OK")

## Cell 2 — List top-level folders in the source container

**What this cell does:** Lists everything at the root of the `source` container — like running `ls` on a folder.

**Line by line:**

- `base_path` — the full root URL of the container. Format: `wasbs://{container}@{account}.blob.core.windows.net/`. The trailing `/` means start from the root.
- `wasbs://` — the protocol for classic Azure Blob Storage. Works with SAS tokens. Do NOT use `abfss://` here — that requires OAuth.
- `dbutils.fs.ls(path)` — Databricks utility to list files and folders at a path. Same idea as `ls` in Linux. Returns a list of `FileInfo` objects, each with `.name`, `.path`, and `.size`.
- `item.size == 0` — folders have a size of 0 bytes. Files have a size greater than 0.
- `item.name` — just the folder or file name, e.g. `"realtime/"`.
- `item.size` — size in bytes.

In [ ]:
base_path = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/"

print(f"Listing root of container: {base_path}\n")

try:
    items = dbutils.fs.ls(base_path)

    for item in items:
        item_type = "DIR " if item.size == 0 else "FILE"
        print(f"  [{item_type}] {item.name:<50} {item.size:>10} bytes")

    print(f"\n  Total: {len(items)} items found at root level")

except Exception as e:
    print(f"ERROR: {e}")
    print("\nCommon causes:")
    print("  403 Forbidden   → SAS token is wrong, expired, or missing 'l' (list) permission")
    print("                    Check: sp=rl should be in your SAS token string")
    print("  Auth error      → Spark SAS config not set — re-run Cell 1 first")
    raise

## Cell 3 — Drill into the charging_sessions folder

**What this cell does:** Goes one level deeper into `realtime/charging_sessions/` to see how files are organised (e.g. year → month → day → hour). You need this to build the correct glob pattern in Cell 5.

**Line by line:**

- `sessions_root` — the full path to the `charging_sessions/` subfolder inside the container.
- First `dbutils.fs.ls(sessions_root)` — lists Level 1 contents, e.g. year folders like `2026/`.
- Second `dbutils.fs.ls(item.path)` — lists Level 2 contents inside each year folder, e.g. `06/` for June.
- `item.path` — the full path of the item (not just the name), needed to `ls` into it.
- The inner `try/except pass` — if an item is a file (not a folder), `ls()` will fail. We silently skip it.

In [ ]:
sessions_root = (
    f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
    f"/realtime/charging_sessions/"
)

print(f"Drilling into: {sessions_root}\n")

try:
    items = dbutils.fs.ls(sessions_root)
    print("Level 1 — contents of charging_sessions/:")
    for item in items:
        print(f"  {item.name}")

    print("\nLevel 2 — drilling one folder deeper:")
    for item in items:
        try:
            sub_items = dbutils.fs.ls(item.path)
            for s in sub_items:
                print(f"  {item.name}{s.name}")
        except:
            pass

except Exception as e:
    print(f"ERROR: {e}")
    print("→ Folder path may differ. Check Cell 2 output for correct folder names.")
    raise

## Cell 4 — Read one specific CSV file

**What this cell does:** Reads a single CSV file into a Spark DataFrame and shows its schema and first 10 rows.

**Line by line:**

- `csv_path` — the full `wasbs://` URL pointing to one specific file. Adjust the date/hour based on what Cell 3 showed.
- `spark.read` — entry point to load data into Spark. You chain options and a format method onto it.
- `.option("header", "true")` — row 1 of the CSV is column names, not a data row.
- `.option("inferSchema", "true")` — Spark scans the file to auto-detect column types (integer, double, string, date, etc.). Without this, every column is treated as plain text (StringType).
- `.csv(csv_path)` — tells Spark the file format is CSV and gives it the path. Returns a DataFrame.
- `df` — a Spark DataFrame: a distributed table held across cluster workers. Nothing is actually read yet — Spark is lazy and waits until you trigger an action.
- `df.count()` — triggers the actual file read. Counts all rows across Spark workers.
- `df.printSchema()` — prints column names and their detected types in a tree view.
- `display(df.limit(10))` — Databricks helper that shows the first 10 rows as an interactive table. `.limit(10)` means only fetch 10 rows — not the whole file.

> Adjust the date/hour in the path based on what Cell 3 showed in your container.

In [ ]:
csv_path = (
    f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
    f"/realtime/charging_sessions/2026/06/01/06/sessions_20260601_0600.csv"
)

print(f"Reading CSV:\n  {csv_path}\n")

try:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(csv_path)
    )

    print(f"Row count    : {df.count():,}")
    print(f"Column count : {len(df.columns)}")
    print(f"Columns      : {df.columns}\n")
    df.printSchema()
    display(df.limit(10))

except Exception as e:
    print(f"ERROR: {e}")
    print("\nCommon causes:")
    print("  'No such file'  → Date/hour in path does not match actual files. Check Cell 3 output.")
    print("  '403 Forbidden' → SAS token missing Read permission (sp must contain 'r')")
    print("  'abfss error'   → You used abfss:// instead of wasbs:// — change the protocol")
    raise

## Cell 5 — Read ALL CSV files at once (glob pattern)

**What this cell does:** Reads every CSV across all year/month/day/hour subfolders in one Spark job using a glob pattern.

**What is a glob pattern?**
A `*` in a path means "match anything at this level". Examples:
- `charging_sessions/*` → any year folder (2025, 2026, ...)
- `charging_sessions/*/*` → any year/month combination
- `charging_sessions/*/*/*/*/*.csv` → any CSV file exactly 4 levels deep

**Line by line:**

- `base` — the root URL (account + container), without any subfolder path yet.
- `glob_path` — the full path with wildcards. The four `/*` represent: year / month / day / hour. The final `/*.csv` matches any CSV file at that depth.
- `spark.read.csv(glob_path)` — Spark finds all matching files in parallel across cluster workers, reads them, and automatically unions them into one single DataFrame.
- `df_all` — one combined DataFrame containing rows from all matched files.
- `df_all.count()` — triggers the actual read and union. Before this line, nothing has been read yet (Spark is lazy).

> If you get a folder-depth error: count the levels between `charging_sessions/` and the `.csv` files in Cell 3 output, then adjust the number of `/*` wildcards accordingly.

In [ ]:
base      = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
glob_path = f"{base}/realtime/charging_sessions/*/*/*/*/*.csv"

print(f"Reading all CSVs using glob pattern:")
print(f"  {glob_path}\n")

try:
    df_all = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(glob_path)
    )

    total_rows = df_all.count()
    print(f"Total rows across all files : {total_rows:,}")
    print(f"Column count                : {len(df_all.columns)}")
    print(f"Columns                     : {df_all.columns}\n")
    display(df_all.limit(10))

except Exception as e:
    print(f"ERROR: {e}")
    print("\nTroubleshooting:")
    print("  1. Run Cell 3 first — check the exact folder depth (year/month/day/hour)")
    print("  2. Count the folder levels between charging_sessions/ and the .csv files")
    print("     3 levels deep → /*/*/*/*.csv")
    print("     4 levels deep → /*/*/*/*/*.csv  (current setting)")
    print("  3. Run Cell 4 with one known file first to confirm basic access is working")
    raise

## Cell 6 — Data quality check

**What this cell does:** Checks every column for null and empty values. Flags columns where more than 10% of rows are missing.

**Line by line:**

- `import pyspark.sql.functions as F` — imports Spark's built-in column functions. `F` is the standard alias — everyone writes it this way.
- `F.col(col_name)` — refers to a column in the DataFrame by name. This does not read data yet — it is just a reference.
- `.isNull()` — returns True/False per row: True if the value is completely missing (NULL).
- `.cast("string")` — temporarily converts the column value to text so we can check if it equals `""` (empty string), even for numeric columns.
- `|` — the OR operator. The filter flags a row if EITHER condition is true: null OR empty string.
- `.filter(condition)` — keeps only rows where the condition is True. Same as SQL `WHERE`.
- `.count()` — counts those filtered rows. This triggers actual Spark computation.
- `pct > 10` — any column where more than 10% of rows are null or empty gets flagged for investigation.

In [ ]:
import pyspark.sql.functions as F

print("=== Data Quality Check — Source Blob ===\n")

try:
    total = df_all.count()
    print(f"Total rows : {total:,}")
    print(f"Columns    : {len(df_all.columns)}\n")

    print(f"  {'Column':<35} {'Nulls':>8}  {'%':>6}  Flag")
    print("  " + "-" * 60)

    for col_name in df_all.columns:
        null_count = df_all.filter(
            F.col(col_name).isNull() | (F.col(col_name).cast("string") == "")
        ).count()

        pct  = (null_count / total * 100) if total > 0 else 0
        flag = "<-- check this" if pct > 10 else ""
        print(f"  {col_name:<35} {null_count:>8,}  {pct:>5.1f}%  {flag}")

    print("\nData quality check complete.")
    print("Silver layer (Day 7) will handle cleaning these nulls.")

except NameError:
    print("ERROR: df_all not found. Run Cell 5 first.")
except Exception as e:
    print(f"ERROR: {e}")

## Cell 7 — Final summary

In [ ]:
print("=" * 55)
print("SOURCE BLOB READ TEST — SUMMARY")
print("=" * 55)
print(f"  Storage account : {STORAGE_ACCOUNT}")
print(f"  Container       : {CONTAINER}")
print(f"  Protocol        : wasbs:// (SAS token auth)")
print(f"  Access          : Read + List only (sp=rl)")
print(f"  Total rows read : {df_all.count():,}")
print(f"  Columns         : {len(df_all.columns)}")
print("=" * 55)
print("SAS token read test PASSED — source blob is accessible.")
print("\nNext: proceed to Day 2.")